In [5]:
# Week5 — Model Validation & DL (BTS data only, Jupyter-friendly)
# Outputs -> ./bts_on_time_data/eda/week5_*.*
# Fast by default; no external datasets are used.

# =========================
# Config
# =========================
ROOT_DIR   = "./bts_on_time_data"
SAMPLE_REL = "eda/sample_100k_week2_features.parquet"
SEED       = 42

# CV folds (temporal, by month)
N_SPLITS   = 4

# Classification cost curve (for threshold selection on each fold)
FP_COST    = 1.0
FN_COST    = 5.0

# DL training (small, CPU/MPS friendly)
DL_EPOCHS  = 20
DL_BATCH   = 512
DL_VAL_SPLIT = 0.15
DL_EARLY_STOP = True
CALIB_NBINS = 15

# =========================
# Imports
# =========================
from pathlib import Path
import json, warnings, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    roc_auc_score, average_precision_score,
    precision_recall_curve, confusion_matrix,
    brier_score_loss
)

from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.utils import check_random_state

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

# Optional TensorFlow for DL demo
try:
    import tensorflow as tf
    TF_OK = True
except Exception:
    TF_OK = False

# =========================
# Paths / IO
# =========================
ROOT = Path(ROOT_DIR); EDA = ROOT/"eda"; EDA.mkdir(parents=True, exist_ok=True)
FEAT = ROOT/SAMPLE_REL
assert FEAT.exists(), f"Not found: {FEAT}"

# =========================
# Helpers
# =========================
def ensure_datetime(df):
    if "FlightDate" in df.columns and not np.issubdtype(df["FlightDate"].dtype, np.datetime64):
        df["FlightDate"] = pd.to_datetime(df["FlightDate"])
    return df

def carrier_col_of(df):
    for c in ["Reporting_Airline","IATA_CODE_Reporting_Airline","UniqueCarrier"]:
        if c in df.columns: return c
    raise KeyError("No carrier column (e.g., Reporting_Airline) found.")

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def metrics_reg(y_true, y_pred):
    return {
        "RMSE": rmse(y_true, y_pred),
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "R2":  float(r2_score(y_true, y_pred)),
    }

def best_f1_threshold(y_true, y_prob):
    p, r, t = precision_recall_curve(y_true, y_prob)
    f1 = 2*(p*r)/(p+r+1e-12)
    i = int(np.nanargmax(f1))
    thr = float(t[i-1]) if i>0 else 0.5
    return thr, float(f1[i]), float(p[i]), float(r[i])

def confusion_at(y_true, y_prob, thr):
    yhat = (y_prob >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, yhat).ravel()
    return {"threshold": float(thr), "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn)}

def business_cost(y_true, y_prob, thr, fp_cost=1.0, fn_cost=5.0):
    cm = confusion_at(y_true, y_prob, thr)
    return fp_cost*cm["FP"] + fn_cost*cm["FN"], cm

def save_pr_curve(y_true, y_prob, out_png, title):
    P,R,_ = precision_recall_curve(y_true, y_prob)
    plt.figure(); plt.plot(R,P); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=120); plt.close()

def save_reliability(y_true, y_prob, out_png, n_bins=15, title="Reliability"):
    # quantile bins
    q = np.linspace(0,1,n_bins+1)
    bins = np.quantile(y_prob, q)
    bins[0], bins[-1] = 0.0, 1.0
    idx = np.digitize(y_prob, bins) - 1
    obs, pred = [], []
    for b in range(n_bins):
        m = (idx==b)
        if np.sum(m) == 0: 
            continue
        obs.append(np.mean(y_true[m]))
        pred.append(np.mean(y_prob[m]))
    plt.figure()
    plt.plot([0,1],[0,1],"--")
    if len(pred)>0:
        plt.plot(pred, obs, "o-")
    plt.xlabel("Predicted probability"); plt.ylabel("Observed frequency"); plt.title(title)
    plt.tight_layout(); plt.savefig(out_png, dpi=120); plt.close()

def safe_numeric(s):
    return pd.to_numeric(s, errors="coerce")

# =========================
# Load & basic splits
# =========================
df = pd.read_parquet(FEAT)
df = ensure_datetime(df)
carrier_col = carrier_col_of(df)

# minimal required cols
need = {"Year","Month","FlightDate","Origin","Dest","Distance","CRSDepTime_min",
        "dow","is_weekend","quarter","season","tod_bin","ArrDelay","ArrDel15"}
miss = need - set(df.columns)
if miss:
    raise ValueError(f"Missing required columns: {sorted(miss)}")

# Targets
df["ArrDel15"] = safe_numeric(df["ArrDel15"]).fillna(0).astype(int)
df = df.dropna(subset=["ArrDelay"])  # keep rows with regression target

# Feature spaces (Week4-compatible + light)
X_num_core = ["Distance","CRSDepTime_min","dow","is_weekend","quarter"]
X_num_opt  = []
for c in ["DepDelay","TaxiOut"]:
    if c in df.columns: X_num_opt.append(c)
X_num = X_num_core + X_num_opt
X_cat = [carrier_col, "Origin","Dest","tod_bin","season"]

y_reg = "ArrDelay"
y_clf = "ArrDel15"

# Preprocessor
def make_preproc(num_cols, cat_cols):
    num_pipe = Pipeline([("imp", SimpleImputer(strategy="median")),
                         ("sc",  StandardScaler())])
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)
    cat_pipe = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                         ("oh",  ohe)])
    return ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("cat", cat_pipe, cat_cols)
    ], remainder="drop")

# Temporal folds (no leakage): sort by FlightDate, split by month blocks
def temporal_folds(df, n_splits=4):
    months = (df[["Year","Month","FlightDate"]]
              .drop_duplicates()[["Year","Month"]]
              .drop_duplicates().sort_values(["Year","Month"]))
    month_pairs = list(months.itertuples(index=False, name=None))
    if len(month_pairs) < n_splits+1:
        # fallback to simple split on index
        n = len(df)
        idx = np.arange(n)
        cuts = np.linspace(0,n,n_splits+1).astype(int)
        for k in range(n_splits):
            tr = idx[:cuts[k+1]]
            va = idx[cuts[k+1]:cuts[min(k+2, len(cuts)-1)]]
            if len(va)==0: break
            yield tr, va
    else:
        # rolling: train up to month k-1, validate on month k
        # pick last n_splits months for validation (ensures enough train)
        val_months = month_pairs[-n_splits:]
        for (yy,mm) in val_months:
            m_train = ( (df["Year"] < yy) | ((df["Year"]==yy) & (df["Month"] < mm)) )
            m_val   = ( (df["Year"]==yy) & (df["Month"]==mm) )
            tr_idx = df.index[m_train].to_numpy()
            va_idx = df.index[m_val].to_numpy()
            if len(va_idx) == 0 or len(tr_idx) == 0:
                continue
            yield tr_idx, va_idx

# =========================
# CV — Regression (Ridge, RF)
# =========================
cv_rows_reg = []
preproc_reg = make_preproc(X_num, X_cat)
ridge = Ridge(alpha=10.0, random_state=SEED)
rfreg = RandomForestRegressor(n_estimators=300, random_state=SEED, n_jobs=-1)

for model_name, mdl in [("ridge", ridge), ("rf", rfreg)]:
    for i, (tr, va) in enumerate(temporal_folds(df, N_SPLITS), 1):
        Xtr = df.loc[tr, X_num+X_cat]; ytr = df.loc[tr, y_reg]
        Xva = df.loc[va, X_num+X_cat]; yva = df.loc[va, y_reg]
        pipe = Pipeline([("prep", preproc_reg), ("mdl", mdl)])
        pipe.fit(Xtr, ytr)
        pred = pipe.predict(Xva)
        m = metrics_reg(yva, pred)
        cv_rows_reg.append({
            "model": model_name, "fold": i, **m, "n_val": len(va)
        })

cv_reg = pd.DataFrame(cv_rows_reg)
cv_reg.to_csv(EDA/"week5_cv_regression.csv", index=False)

# =========================
# CV — Classification (LogReg, RF)
# =========================
cv_rows_clf = []
preproc_clf = make_preproc(X_num, X_cat)
logreg = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
rfclf  = RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                random_state=SEED, n_jobs=-1)

for model_name, mdl in [("logreg", logreg), ("rf", rfclf)]:
    for i, (tr, va) in enumerate(temporal_folds(df, N_SPLITS), 1):
        Xtr = df.loc[tr, X_num+X_cat]; ytr = df.loc[tr, y_clf]
        Xva = df.loc[va, X_num+X_cat]; yva = df.loc[va, y_clf]
        pipe = Pipeline([("prep", preproc_clf), ("mdl", mdl)])
        pipe.fit(Xtr, ytr)
        prob = pipe.predict_proba(Xva)[:,1]
        roc = float(roc_auc_score(yva, prob))
        pra = float(average_precision_score(yva, prob))
        thr, f1b, pb, rb = best_f1_threshold(yva, prob)
        cm05 = confusion_at(yva, prob, 0.5)
        cmb  = confusion_at(yva, prob, thr)
        thrs = np.linspace(0.01,0.99,199)
        costs = [business_cost(yva, prob, t, FP_COST, FN_COST)[0] for t in thrs]
        thr_cost = float(thrs[int(np.argmin(costs))])
        brier = float(brier_score_loss(yva, prob))
        # save fold curves (optional)
        base = f"week5_cv_{model_name}_fold{i}"
        pd.DataFrame({"thr":thrs,"val_cost":costs}).to_csv(EDA/f"{base}_cost.csv", index=False)
        save_pr_curve(yva, prob, EDA/f"{base}_pr.png", f"PR — {model_name} fold{i}")
        save_reliability(yva, prob, EDA/f"{base}_reliability.png", n_bins=CALIB_NBINS,
                         title=f"Reliability — {model_name} fold{i}")
        cv_rows_clf.append({
            "model": model_name, "fold": i,
            "ROC_AUC": roc, "PR_AUC": pra, "best_thr_F1": thr, "best_F1": f1b,
            "prec@best": pb, "rec@best": rb, "brier": brier,
            "cm@0.5_TP": cm05["TP"], "cm@0.5_FP": cm05["FP"],
            "cm@0.5_TN": cm05["TN"], "cm@0.5_FN": cm05["FN"],
            "cm@best_TP": cmb["TP"], "cm@best_FP": cmb["FP"],
            "cm@best_TN": cmb["TN"], "cm@best_FN": cmb["FN"],
            "thr_cost": thr_cost, "n_val": len(yva)
        })

cv_clf = pd.DataFrame(cv_rows_clf)
cv_clf.to_csv(EDA/"week5_cv_classification.csv", index=False)

# =========================
# DL demo — TensorFlow MLP (classification on BTS)
# Temporal split: Train=2024, Val=2025-Jan..Mar, Test=2025-Apr
# =========================
dl_summary = {}
if TF_OK:
    # temporal split
    tr = df["Year"]==2024
    va = (df["Year"]==2025) & (df["Month"].isin([1,2,3]))
    te = (df["Year"]==2025) & (df["Month"]==4)

    # fit preprocessor on train only
    pre_clf = make_preproc(X_num, X_cat)
    Xtr = pre_clf.fit_transform(df.loc[tr, X_num+X_cat])
    Xva = pre_clf.transform(df.loc[va, X_num+X_cat])
    Xte = pre_clf.transform(df.loc[te, X_num+X_cat])
    ytr = df.loc[tr, y_clf].to_numpy().astype("float32")
    yva = df.loc[va, y_clf].to_numpy().astype("float32")
    yte = df.loc[te, y_clf].to_numpy().astype("float32")

    # small MLP
    tf.random.set_seed(SEED)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(Xtr.shape[1],)),
        tf.keras.layers.Dense(128, activation="relu"),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid")
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
                  loss="binary_crossentropy",
                  metrics=[tf.keras.metrics.AUC(curve="ROC", name="roc"),
                           tf.keras.metrics.AUC(curve="PR", name="pr")])
    cb = []
    if DL_EARLY_STOP:
        cb.append(tf.keras.callbacks.EarlyStopping(
            monitor="val_pr", mode="max", patience=3, restore_best_weights=True
        ))
    hist = model.fit(
        Xtr, ytr, epochs=DL_EPOCHS, batch_size=DL_BATCH,
        validation_data=(Xva, yva), verbose=0, callbacks=cb
    )
    # evaluate on TEST
    prob_te = model.predict(Xte, batch_size=4096, verbose=0).squeeze()
    roc = float(roc_auc_score(yte, prob_te))
    pra = float(average_precision_score(yte, prob_te))
    brier = float(brier_score_loss(yte, prob_te))
    thr, f1b, pb, rb = best_f1_threshold(yva, model.predict(Xva, batch_size=4096, verbose=0).squeeze())
    cm05 = confusion_at(yte, prob_te, 0.5)
    cmb  = confusion_at(yte, prob_te, thr)

    # plots
    plt.figure()
    plt.plot(hist.history["roc"]); plt.plot(hist.history["val_roc"])
    plt.legend(["roc","val_roc"]); plt.title("TF MLP ROC-AUC"); plt.tight_layout()
    plt.savefig(EDA/"week5_tf_training_roc.png", dpi=120); plt.close()

    plt.figure()
    plt.plot(hist.history["pr"]); plt.plot(hist.history["val_pr"])
    plt.legend(["pr","val_pr"]); plt.title("TF MLP PR-AUC"); plt.tight_layout()
    plt.savefig(EDA/"week5_tf_training_pr.png", dpi=120); plt.close()

    save_pr_curve(yte, prob_te, EDA/"week5_tf_pr_test.png", "PR — TF MLP (TEST)")
    save_reliability(yte, prob_te, EDA/"week5_tf_reliability_test.png", CALIB_NBINS,
                     "Reliability — TF MLP (TEST)")

    pd.DataFrame({"y_true": yte, "y_prob": prob_te}).to_csv(EDA/"week5_tf_prob_test.csv", index=False)

    dl_summary = {
        "tf_mlp_test": {"ROC_AUC": roc, "PR_AUC": pra, "Brier": brier,
                        "cm@0.5": cm05, "cm@best": cmb, "best_thr_from_val": float(thr)}
    }
else:
    print("[INFO] TensorFlow not available; skip DL demo.")

# =========================
# Save overall summary (JSON) + human-readable console digest
# =========================
summary = {
    "cv_regression_csv": str(EDA/"week5_cv_regression.csv"),
    "cv_classification_csv": str(EDA/"week5_cv_classification.csv"),
    "dl": dl_summary,
    "config": {
        "n_splits": N_SPLITS, "fp_cost": FP_COST, "fn_cost": FN_COST,
        "dl_epochs": DL_EPOCHS, "dl_batch": DL_BATCH
    },
    "features": {"num": X_num, "cat": X_cat, "y_reg": y_reg, "y_clf": y_clf}
}
with open(EDA/"week5_metrics.json","w") as f:
    json.dump(summary, f, indent=2)

def fmt(x):
    try: return f"{float(x):.3f}"
    except: return str(x)

print("\n=== Week5 — CV Regression (temporal folds) ===")
if not cv_reg.empty:
    bym = cv_reg.groupby("model")[["RMSE","MAE","R2"]].mean().reset_index()
    for _,r in bym.iterrows():
        print(f"{r['model']:>6s} :: RMSE={fmt(r['RMSE'])}  MAE={fmt(r['MAE'])}  R2={fmt(r['R2'])}")

print("\n=== Week5 — CV Classification (temporal folds) ===")
if not cv_clf.empty:
    bym = cv_clf.groupby("model")[["ROC_AUC","PR_AUC","brier"]].mean().reset_index()
    for _,r in bym.iterrows():
        print(f"{r['model']:>6s} :: ROC_AUC={fmt(r['ROC_AUC'])}  PR_AUC={fmt(r['PR_AUC'])}  Brier={fmt(r['brier'])}")

if dl_summary:
    t = dl_summary["tf_mlp_test"]
    print("\n=== Week5 — DL (TF MLP on TEST) ===")
    print(f"ROC_AUC={fmt(t['ROC_AUC'])}  PR_AUC={fmt(t['PR_AUC'])}  Brier={fmt(t['Brier'])}")
    print(f"cm@0.5={t['cm@0.5']}  cm@best={t['cm@best']}  best_thr={fmt(t['best_thr_from_val'])}")

print(f"\n[DONE] Summary JSON -> {EDA/'week5_metrics.json'}")
print("Artifacts saved with prefix week5_* under ./bts_on_time_data/eda")


=== Week5 — CV Regression (temporal folds) ===
    rf :: RMSE=11.740  MAE=8.366  R2=0.956
 ridge :: RMSE=11.236  MAE=8.022  R2=0.960

=== Week5 — CV Classification (temporal folds) ===
logreg :: ROC_AUC=0.962  PR_AUC=0.922  Brier=0.065
    rf :: ROC_AUC=0.956  PR_AUC=0.905  Brier=0.054

=== Week5 — DL (TF MLP on TEST) ===
ROC_AUC=0.961  PR_AUC=0.920  Brier=0.045
cm@0.5={'threshold': 0.5, 'TP': 1259, 'FP': 99, 'TN': 6537, 'FN': 373}  cm@best={'threshold': 0.4480578601360321, 'TP': 1287, 'FP': 115, 'TN': 6521, 'FN': 345}  best_thr=0.448

[DONE] Summary JSON -> bts_on_time_data/eda/week5_metrics.json
Artifacts saved with prefix week5_* under ./bts_on_time_data/eda


In [6]:
import json, os, glob
import pandas as pd
from pathlib import Path

EDA = Path("bts_on_time_data/eda")
cv_reg = EDA/"week5_cv_regression.csv"
cv_clf = EDA/"week5_cv_classification.csv"
meta   = EDA/"week5_metrics.json"

def fmt(x): 
    try: return f"{float(x):.3f}"
    except: return str(x)

print("=== Week5 — Summary (auto) ===")
if cv_reg.exists():
    d = pd.read_csv(cv_reg)
    g = d.groupby("model")[["RMSE","MAE","R2"]].mean().reset_index()
    for _,r in g.iterrows():
        print(f"[REG] {r['model']:>6s} :: RMSE={fmt(r['RMSE'])}  MAE={fmt(r['MAE'])}  R2={fmt(r['R2'])}")
if cv_clf.exists():
    d = pd.read_csv(cv_clf)
    g = d.groupby("model")[["ROC_AUC","PR_AUC","brier"]].mean().reset_index()
    for _,r in g.iterrows():
        print(f"[CLF] {r['model']:>6s} :: ROC_AUC={fmt(r['ROC_AUC'])}  PR_AUC={fmt(r['PR_AUC'])}  Brier={fmt(r['brier'])}")
dl_line = ""
if meta.exists():
    m = json.loads(meta.read_text())
    if m.get("dl",{}):
        t = m["dl"]["tf_mlp_test"]
        dl_line = f"[DL] TF-MLP TEST :: ROC_AUC={fmt(t['ROC_AUC'])}  PR_AUC={fmt(t['PR_AUC'])}  Brier={fmt(t['Brier'])}  best_thr={fmt(t['best_thr_from_val'])}"
        print(dl_line)

# build markdown
md = []
md += ["# Week5 Report — Files & Findings / Week5 报告 — 文件与结论", ""]
if not cv_reg.exists() and not cv_clf.exists():
    md += ["No week5_* files found. / 未发现 week5_* 文件。", ""]
else:
    if cv_reg.exists():
        md += ["## Regression / 回归", f"- Table 表: 「{cv_reg.name}」", ""]
    if cv_clf.exists():
        md += ["## Classification / 分类", f"- Table 表: 「{cv_clf.name}」", ""]
    # per-fold assets
    for model in ["logreg","rf"]:
        files = sorted(glob.glob(str(EDA/f"week5_cv_{model}_fold*_*.*")))
        if files:
            md += [f"### {model} folds / {model} 各折", ""]
            for f in files:
                md += [f"- Figure/CSV: 「{Path(f).name}」", f"- 图/表： 「{Path(f).name}」", ""]
    # DL assets
    dl_files = [
        "week5_tf_training_roc.png","week5_tf_training_pr.png",
        "week5_tf_pr_test.png","week5_tf_reliability_test.png",
        "week5_tf_prob_test.csv"
    ]
    has_dl = False
    for f in dl_files:
        if (EDA/f).exists():
            has_dl = True; break
    if has_dl:
        md += ["## Deep Learning / 深度学习", ""]
        for f in dl_files:
            p = EDA/f
            if p.exists():
                md += [f"- Artifact: 「{p.name}」", f"- 产物： 「{p.name}」", ""]
    if meta.exists():
        md += ["## Summary JSON / 汇总 JSON", f"- File: 「{meta.name}」", f"- 文件： 「{meta.name}」", ""]
        if dl_line: md += ["", dl_line, dl_line.replace(" :: ","： ")]

out = EDA/"week5_report.md"
out.write_text("\n".join(md), encoding="utf-8")
print(f"\n[OK] Wrote markdown -> {out}")

=== Week5 — Summary (auto) ===
[REG]     rf :: RMSE=11.740  MAE=8.366  R2=0.956
[REG]  ridge :: RMSE=11.236  MAE=8.022  R2=0.960
[CLF] logreg :: ROC_AUC=0.962  PR_AUC=0.922  Brier=0.065
[CLF]     rf :: ROC_AUC=0.956  PR_AUC=0.905  Brier=0.054
[DL] TF-MLP TEST :: ROC_AUC=0.961  PR_AUC=0.920  Brier=0.045  best_thr=0.448

[OK] Wrote markdown -> bts_on_time_data/eda/week5_report.md


======== Week5 Extract — Files & Stats ========

======== Regression / 回归 ========
Table / 表: week5_cv_regression.csv
[week5_cv_regression.csv] table missing or empty.
Full table / 全表：
(missing)

======== Classification / 分类 ========
Table / 表: week5_cv_classification.csv
[week5_cv_classification.csv] table missing or empty.
Full table / 全表：
(missing)

======== Per-fold artifacts — Logistic Regression / 各折产物 — 逻辑回归 ========
PR curves (logreg) / PR 曲线（logreg） (0 files)
Reliability (logreg) / 可靠性（logreg） (0 files)

======== Per-fold artifacts — Random Forest / 各折产物 — 随机森林 ========
PR curves (rf) / PR 曲线（rf） (0 files)
Reliability (rf) / 可靠性（rf） (0 files)

======== Deep Learning (TensorFlow) / 深度学习（TensorFlow） ========
No week5_tf_prob_test.csv found / 未发现 week5_tf_prob_test.csv
Training ROC / 训练 ROC (0 files)
Training PR / 训练 PR (0 files)
Test PR / 测试 PR (0 files)
Test Reliability / 测试可靠性 (0 files)

======== Summary JSON / 汇总 JSON ========
No week5_metrics.json found / 未发现 week5_metrics.j

In [8]:
import os, sys, json, glob, io, argparse
from pathlib import Path
import pandas as pd
import numpy as np

def find_base_dir(cli_base: str | None) -> Path:
    # 1) CLI --base 优先
    if cli_base:
        p = Path(cli_base).expanduser().resolve()
        if p.exists():
            return p
    # 2) 常见默认
    for cand in ["bts_on_time_data/eda", "./bts_on_time_data/eda", "eda", "."]:
        p = Path(cand).expanduser().resolve()
        if p.exists():
            # 如果里面确实有 week5_* 文件就用它
            if list(p.glob("week5_*")):
                return p
    # 3) 递归搜索整个仓库，找包含最多 week5_* 的目录
    repo_root = Path(".").resolve()
    hits = {}
    for fp in repo_root.rglob("week5_*"):
        if fp.is_file():
            hits.setdefault(fp.parent, 0)
            hits[fp.parent] += 1
    if hits:
        return max(hits, key=hits.get)
    # 兜底：当前目录
    return repo_root

def line(s): buf.append(s)
def hr(title): line("\n" + "="*8 + " " + title + " " + "="*8)
def try_read_csv(path, **kw):
    try: return pd.read_csv(path, **kw)
    except Exception: return None
def print_full_table(df, max_cols=None):
    if df is None:
        line("(missing)"); return
    with pd.option_context("display.max_rows", None,
                           "display.max_columns", max_cols or None,
                           "display.width", 200,
                           "display.max_colwidth", 200):
        line(df.to_string(index=False))
def fmt_float(x):
    try: return f"{float(x):.6g}"
    except: return str(x)

def summarize_regression_table(df, title):
    if df is None or df.empty:
        line(f"[{title}] table missing or empty."); return
    line(f"[{title}] rows={len(df)} cols={len(df.columns)}")
    lower = [c.lower() for c in df.columns]
    c_model = df.columns[lower.index("model")] if "model" in lower else None
    c_rmse  = next((df.columns[i] for i,n in enumerate(lower) if n in ("rmse","val_rmse","test_rmse")), None)
    c_mae   = next((df.columns[i] for i,n in enumerate(lower) if n in ("mae","val_mae","test_mae")), None)
    c_r2    = next((df.columns[i] for i,n in enumerate(lower) if n in ("r2","val_r2","test_r2")), None)
    if c_model and c_rmse and c_mae and c_r2:
        g = df.groupby(c_model).agg({c_rmse:"mean", c_mae:"mean", c_r2:"mean"}).reset_index()
        line("Per-model mean metrics / 各模型均值：")
        print_full_table(g)
    else:
        line("Columns not standard; printing full table / 列名非常规，直接打印全表：")
        print_full_table(df)

def summarize_classification_table(df, title):
    if df is None or df.empty:
        line(f"[{title}] table missing or empty."); return
    line(f"[{title}] rows={len(df)} cols={len(df.columns)}")
    lower = [c.lower() for c in df.columns]
    c_model = df.columns[lower.index("model")] if "model" in lower else None
    auc_cols = [df.columns[i] for i,n in enumerate(lower) if n in ("roc_auc","pr_auc")]
    c_brier = next((df.columns[i] for i,n in enumerate(lower) if n=="brier"), None)
    if c_model and (auc_cols or c_brier):
        agg = {c:"mean" for c in auc_cols}
        if c_brier: agg[c_brier] = "mean"
        g = df.groupby(c_model).agg(agg).reset_index()
        line("Per-model mean metrics / 各模型均值：")
        print_full_table(g)
    else:
        line("Columns not standard; printing full table / 列名非常规，直接打印全表：")
        print_full_table(df)

def summarize_cost_csv(path: Path):
    df = try_read_csv(path)
    line(f"- File / 文件: {path.name}")
    if df is None or df.empty:
        line("  (missing/empty)"); return
    lower = [c.lower() for c in df.columns]
    c_thr  = next((df.columns[i] for i,n in enumerate(lower) if n in ("threshold","thr","thresh")), None)
    c_cost = next((df.columns[i] for i,n in enumerate(lower) if n=="cost"), None)
    if c_thr and c_cost:
        idx = df[c_cost].idxmin()
        row = df.loc[idx]
        line("  Min-cost row / 最小成本行：")
        line("  " + ", ".join(f"{col}={fmt_float(row[col])}" for col in df.columns))
    else:
        line("  Columns not standard; head(10) / 列名非常规，展示前10行：")
        print_full_table(df.head(10))

def summarize_prob_csv(path: Path, max_head=10, max_tail=10):
    df = try_read_csv(path)
    line(f"- File / 文件: {path.name}")
    if df is None or df.empty:
        line("  (missing/empty)"); return
    lower = [c.lower() for c in df.columns]
    c_prob = next((df.columns[i] for i,n in enumerate(lower) if n in ("prob","probability","p","pred_prob","y_prob")), None)
    c_true = next((df.columns[i] for i,n in enumerate(lower) if n in ("label","y_true","y","target","truth")), None)
    line(f"  rows={len(df)} cols={len(df.columns)}")
    if c_prob:
        p = pd.to_numeric(df[c_prob], errors="coerce").dropna()
        q = np.quantile(p, [0,0.25,0.5,0.75,1.0]) if len(p)>0 else [np.nan]*5
        line("  prob five-number summary / 概率五数概括：")
        line("  " + " ".join([f"{v:.4f}" if pd.notna(v) else "nan" for v in q]))
    line("  head(10):"); print_full_table(df.head(max_head))
    line("  tail(10):"); print_full_table(df.tail(max_tail))

def list_artifacts(base: Path, pattern: str, title: str):
    files = sorted(base.glob(pattern))
    line(f"{title} ({len(files)} files)")
    for f in files: line(" - " + f.name)
    return files

# -------- main --------
# 忽略 notebook 注入参数，用 argparse 只取我们认识的 --base
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument("--base", type=str, default=None)
args, _ = parser.parse_known_args()

BASE = find_base_dir(args.base)
OUTF = BASE / "week5_extract.txt"

buf = []
hr(f"Week5 Extract — Files & Stats @ {BASE}")

# 1) Regression
reg = try_read_csv(BASE / "week5_cv_regression.csv")
hr("Regression / 回归")
line(f"Table / 表: {(BASE / 'week5_cv_regression.csv').name}")
summarize_regression_table(reg, "week5_cv_regression.csv")
line("Full table / 全表："); print_full_table(reg)

# 2) Classification
clf = try_read_csv(BASE / "week5_cv_classification.csv")
hr("Classification / 分类")
line(f"Table / 表: {(BASE / 'week5_cv_classification.csv').name}")
summarize_classification_table(clf, "week5_cv_classification.csv")
line("Full table / 全表："); print_full_table(clf)

# 3) Per-fold — logreg
hr("Per-fold artifacts — Logistic Regression / 各折产物 — 逻辑回归")
for p in sorted(BASE.glob("week5_cv_logreg_fold*_cost.csv")):
    summarize_cost_csv(p)
list_artifacts(BASE, "week5_cv_logreg_fold*_pr.png", "PR curves (logreg) / PR 曲线（logreg）")
list_artifacts(BASE, "week5_cv_logreg_fold*_reliability.png", "Reliability (logreg) / 可靠性（logreg）")

# 4) Per-fold — RF
hr("Per-fold artifacts — Random Forest / 各折产物 — 随机森林")
for p in sorted(BASE.glob("week5_cv_rf_fold*_cost.csv")):
    summarize_cost_csv(p)
list_artifacts(BASE, "week5_cv_rf_fold*_pr.png", "PR curves (rf) / PR 曲线（rf）")
list_artifacts(BASE, "week5_cv_rf_fold*_reliability.png", "Reliability (rf) / 可靠性（rf）")

# 5) Deep Learning
hr("Deep Learning (TensorFlow) / 深度学习（TensorFlow）")
prob = BASE / "week5_tf_prob_test.csv"
if prob.exists(): summarize_prob_csv(prob)
else: line("No week5_tf_prob_test.csv found / 未发现 week5_tf_prob_test.csv")
list_artifacts(BASE, "week5_tf_training_roc.png", "Training ROC / 训练 ROC")
list_artifacts(BASE, "week5_tf_training_pr.png", "Training PR / 训练 PR")
list_artifacts(BASE, "week5_tf_pr_test.png", "Test PR / 测试 PR")
list_artifacts(BASE, "week5_tf_reliability_test.png", "Test Reliability / 测试可靠性")

# 6) Summary JSON
hr("Summary JSON / 汇总 JSON")
jpath = BASE / "week5_metrics.json"
if jpath.exists():
    try:
        j = json.loads(jpath.read_text(encoding="utf-8"))
        line("Raw JSON / 原始 JSON：")
        line(json.dumps(j, ensure_ascii=False, indent=2))
    except Exception as e:
        line(f"Failed to read JSON: {e}")
else:
    line("No week5_metrics.json found / 未发现 week5_metrics.json")

report = "\n".join(buf).strip() + "\n"
OUTF.write_text(report, encoding="utf-8")
print(report)
print(f"\n[Saved] {OUTF}")

======== Week5 Extract — Files & Stats @ /Users/guohaoyang/Desktop/vscworkspace/BTS/bts_on_time_data/eda ========

======== Regression / 回归 ========
Table / 表: week5_cv_regression.csv
[week5_cv_regression.csv] rows=8 cols=6
Per-model mean metrics / 各模型均值：
model      RMSE      MAE       R2
   rf 11.740110 8.365516 0.956306
ridge 11.236228 8.022451 0.959954
Full table / 全表：
model  fold      RMSE      MAE       R2  n_val
ridge     1 11.378337 8.210723 0.959167   7703
ridge     2 11.448538 8.234397 0.960068   7323
ridge     3 11.151830 7.971865 0.964332   8668
ridge     4 10.966209 7.672817 0.956250   8268
   rf     1 11.963887 8.616524 0.954856   7703
   rf     2 11.935000 8.585286 0.956602   7323
   rf     3 11.691400 8.280234 0.960797   8668
   rf     4 11.370152 7.980020 0.952968   8268

======== Classification / 分类 ========
Table / 表: week5_cv_classification.csv
[week5_cv_classification.csv] rows=8 cols=19
Per-model mean metrics / 各模型均值：
 model  ROC_AUC   PR_AUC    brier
logreg 0.9620